# STAT 764 · Lab 2 — Find the leak

**Assigned Tue Sep 15 · Due Tue Sep 22, 11:59 pm**

Individual work. Parts 2 and 3 need Meeting 4 (Thursday Sep 17).

Same rules as Lab 1: `git pull`, copy into `work/` before editing, Restart &
Run All before submitting, upload the `.ipynb` to Canvas.

In [ ]:
import pathlib
import sys

here = pathlib.Path.cwd()
root = next(p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists())
sys.path.insert(0, str(root / "course"))

from stat764 import load

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold, cross_val_score, train_test_split

ames = load("ames.csv")

---

## Part 1 · One split is a random number (15 points)

In Lab 1 you reported a held-out R-squared from a single split with
`random_state=764`. That number was one draw from a distribution.

**(a)** Rebuild your Lab 1 pipeline here (copy it over).

**(b)** Score it on 50 different random splits. Report the mean, the SD, the
minimum and the maximum.

**(c)** Report your 10-fold cross-validated R-squared, with the SD across folds.

**(d)** In the written section: where did your single Lab 1 number fall inside
that distribution — near the middle, or in a tail?

In [ ]:
# YOUR CODE HERE

## Part 2 · A pipeline that is too good (20 points)

Below is a workflow somebody sent you. They are pleased with it: a
cross-validated R-squared over 0.90 on a sample of 200 recent sales, which is
better than anything you built in Lab 1 on the full dataset.

They have added one engineered feature. Local realtors talk about
"blocks" — a neighborhood combined with a building class — so they built a
feature holding the **average sale price of the block each house sits in**.

Run it. Then work out why the number is not real.

In [ ]:
sample = ames.sample(n=200, random_state=764).reset_index(drop=True)
sample["Block"] = sample["Neighborhood"] + "_" + sample["MS_SubClass"].astype(str)

NUMERIC = ["Gr_Liv_Area", "Lot_Area", "Year_Built", "Overall_Qual",
           "Total_Bsmt_SF", "Garage_Cars"]

# the engineered feature: average sale price of this house's block
sample["block_avg_price"] = sample.groupby("Block")["SalePrice"].transform("mean")

X_theirs = sample[NUMERIC + ["block_avg_price"]].fillna(sample[NUMERIC].median())
y_theirs = sample["SalePrice"]

their_score = cross_val_score(LinearRegression(), X_theirs, y_theirs,
                              cv=KFold(5, shuffle=True, random_state=764),
                              scoring="r2").mean()
print(f"  their cross-validated R-squared: {their_score:.4f}")
print(f"  blocks in this sample: {sample['Block'].nunique()}")
print(f"  average sales per block: {len(sample) / sample['Block'].nunique():.1f}")

**(a)** In the written section, explain precisely what information reaches the
model that would not be available when predicting a house that has not sold
yet. Name the line that does it.

**(b)** Build the honest version. The block average is a reasonable feature —
you are not deleting it, you are computing it correctly. It must be computed
**from training rows only**, separately within each fold, and you need a
sensible fallback for a block that appears in a test fold but not in training.

Report the honest cross-validated R-squared and the size of the gap.

In [ ]:
# YOUR CODE HERE
#
# Hint: cross_val_score cannot do this for you, because the feature depends on
# the outcome. Write the fold loop yourself with KFold(...).split(...).

## Part 3 · How big is the leak? (15 points)

The gap you just measured is not a fixed property of the mistake. It depends on
the data.

Re-run **both** versions — leaky and honest — at three sample sizes: **200,
400, and the full 2,930**. Make a small table of the leaky score, the honest
score, the gap, and the average number of sales per block.

**(c)** In the written section: what happens to the gap as the sample grows,
and why? State the relationship in terms of something you can compute from the
data before fitting anything.

In [ ]:
# YOUR CODE HERE

## Part 4 · Written (20 points)

**(d)** Part 1(d): where did your single Lab 1 number fall in the distribution
of 50 splits?

**(a)** Part 2(a): what leaks, and which line does it?

**(c)** Part 3(c): how does the gap depend on the data, and what would you
check *before* fitting to predict whether this mistake would matter?

**(e)** You have now seen a leak worth 0.11 of R-squared and, in Meeting 4, one
worth 0.000. Both are the same category of mistake. How would you decide, on a
real project, which leaks are worth the time to fix?

### Your answers

*(double-click to edit)*

**(d)**

**(a)**

**(c)**

**(e)**

---

## Before you submit

- [ ] **Run → Restart Kernel and Run All Cells**
- [ ] Part 3 reports all three sample sizes
- [ ] All four written answers are filled in
- [ ] Uploaded the `.ipynb` to Canvas

**70 points.**